In [ ]:
!pip install pyspark -q

In [ ]:
import pyspark
print(f"PySpark versão:{pyspark.__version__} ")

PySpark versão:4.0.2 


In [ ]:
from pyspark.sql import SparkSession

# Cria a SparkSession
spark = SparkSession.builder \
    .appName("PagBank-FraudeSentinel") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"Spark rodando! Versão: {spark.version}")


Spark rodando! Versão: 4.0.2


In [ ]:
from decimal import HAVE_CONTEXTVAR
#Lendo o Dataset
df = spark.read.csv("/content/PS_20174392719_1491204439457_log.csv",
header=True, #Mostra a primeira linha do cabeçalho
inferSchema=True) #Spark detecta os tipos automaticamente

#Verifica se carregou corretamente
print(f"Total de linhas: {df.count()}")
print(f"Total de colunas: {len(df.columns)}")
df.printSchema()

Total de linhas: 6362620
Total de colunas: 11
root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



In [ ]:
df.show(5, truncate=False)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|type    |amount  |nameOrig   |oldbalanceOrg|newbalanceOrig|nameDest   |oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|1   |PAYMENT |9839.64 |C1231006815|170136.0     |160296.36     |M1979787155|0.0           |0.0           |0      |0             |
|1   |PAYMENT |1864.28 |C1666544295|21249.0      |19384.72      |M2044282225|0.0           |0.0           |0      |0             |
|1   |TRANSFER|181.0   |C1305486145|181.0        |0.0           |C553264065 |0.0           |0.0           |1      |0             |
|1   |CASH_OUT|181.0   |C840083671 |181.0        |0.0           |C38997010  |21182.0       |0.0           |1      |0             |
|1   |PAYMENT |11668.14|C2048537720|41554.0      |29885.86      |M1230701703|0.0   

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum

# Contando nulos por coluna
print("=== Valores nulos por coluna ===")
df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

=== Valores nulos por coluna ===
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|step|type|amount|nameOrig|oldbalanceOrg|newbalanceOrig|nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|   0|   0|     0|       0|            0|             0|       0|             0|             0|      0|             0|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+



In [ ]:
from pyspark.sql.functions import count, round as spark_round

# Quantas fraudes vs transações legítimas?
print("=== Distribuição de Fraudes ===")
df.groupBy("isFraud") \
  .agg(count("*").alias("total")) \
  .withColumn("percentual", spark_round(col("total") / 6362620 * 100, 4)) \
  .show()

""" Conforme o resultado, o percentual de 0.0019% indica um desbalanceamento de classes, o que
faz o modelo detectar erroneamente 99% dos casos como Sem Fraude.
"""


=== Distribuição de Fraudes ===
+-------+-------+----------+
|isFraud|  total|percentual|
+-------+-------+----------+
|      1|   8213|    0.1291|
|      0|6354407|   99.8709|
+-------+-------+----------+



' Conforme o resultado, o percentual de 0.0019% indica um desbalanceamento de classes, o que\nfaz o modelo detectar erroneamente 99% dos casos como Sem Fraude.\n'

In [ ]:
# Quais tipos de transação concentram as fraudes?
print("=== Fraudes por Tipo de Transação ===")
df.filter(col("isFraud") == 1) \
  .groupBy("type") \
  .agg(count("*").alias("total_fraudes")) \
  .orderBy("total_fraudes", ascending=False) \
  .show()

=== Fraudes por Tipo de Transação ===
+--------+-------------+
|    type|total_fraudes|
+--------+-------------+
|CASH_OUT|         4116|
|TRANSFER|         4097|
+--------+-------------+



In [ ]:
# Filtrando e salvando os dados limpos
from pyspark.sql.functions import when

df_filtrado = df.filter(col("type").isin(["CASH_OUT", "TRANSFER"]))

# Se o saldo não zerou após a transação, pode ser sinal de fraude
df_filtrado = df_filtrado.withColumn(
    "erro_saldo_orig",
    col("newbalanceOrig") - (col("oldbalanceOrg") - col("amount"))
)

print(f"Dataset Filtrado: {df_filtrado.count():,} transações")
print(f"Fraudes no filtrado: {df_filtrado.filter(col('isFraud')==1).count():,}")
df_filtrado.show(5, truncate=False)


Dataset Filtrado: 2,770,409 transações
Fraudes no filtrado: 8,213
+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+---------------+
|step|type    |amount   |nameOrig   |oldbalanceOrg|newbalanceOrig|nameDest   |oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|erro_saldo_orig|
+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+---------------+
|1   |TRANSFER|181.0    |C1305486145|181.0        |0.0           |C553264065 |0.0           |0.0           |1      |0             |0.0            |
|1   |CASH_OUT|181.0    |C840083671 |181.0        |0.0           |C38997010  |21182.0       |0.0           |1      |0             |0.0            |
|1   |CASH_OUT|229133.94|C905080434 |15325.0      |0.0           |C476402209 |5083.0        |51513.44      |0      |0             |213808.94      |
|1   |TRANSFER|215310.3 |C1670993182|705.0    

In [ ]:
# Salvando dados em Parquet (dados colunares) para o Spark ler mais rápido

df_filtrado.write \
.mode("overwrite") \
.partitionBy("type") \
.parquet("/content/fraude_pysim_filtrado")

print("Salvo com sucesso")

# Verificando partições criadas
import os
print(os.listdir('/content/fraude_pysim_filtrado'))

"""Aqui o spark criou 2 pastas automaticamente para cada tipo de transação"""
"""quando um analista precisar consultar só CASH_OUT, o Spark lê apenas aquela pasta — ignora TRANSFER completamente.
Isso se chama partition pruning e acelera queries enormemente em datasets de bilhões de linhas."""

Salvo com sucesso
['._SUCCESS.crc', '_SUCCESS', 'type=CASH_OUT', 'type=TRANSFER']


'quando um analista precisar consultar só CASH_OUT, o Spark lê apenas aquela pasta — ignora TRANSFER completamente. \nIsso se chama partition pruning e acelera queries enormemente em datasets de bilhões de linhas.'

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Recriando a sessão se necessário
spark = SparkSession.builder \
    .appName("PagBank-FraudeSentinel") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# Relendo o CSV original
df = spark.read.csv(
    "/content/PS_20174392719_1491204439457_log.csv",
    header=True,
    inferSchema=True
)

# Refazendo o filtro e a feature
df_spark = df.filter(col("type").isin(["CASH_OUT", "TRANSFER"])) \
             .withColumn(
                 "erro_saldo_orig",
                 col("newbalanceOrig") - (col("oldbalanceOrg") - col("amount"))
             )

# Registrando como tabela SQL
df_spark.createOrReplaceTempView("transacoes")

print("Pronto! Conferindo...")
spark.sql("""
    SELECT type, COUNT(*) as total, SUM(isFraud) as fraudes
    FROM transacoes
    GROUP BY type
    ORDER BY fraudes DESC
""").show()

Pronto! Conferindo...
+--------+-------+-------+
|    type|  total|fraudes|
+--------+-------+-------+
|CASH_OUT|2237500|   4116|
|TRANSFER| 532909|   4097|
+--------+-------+-------+



In [ ]:
# Detectando comportamentos suspeitos com o Windows Functions

from pyspark.sql.functions import count, avg, max as spark_max, sum as spark_sum
from pyspark.sql.window import Window

# Definindo janelas de análise por conta
janela_origem = Window.partitionBy("nameOrig")
janela_destino = Window.partitionBy("nameDest")

df_features = df_spark \
    .withColumn(
        "qtd_transacoes_origem",
        count("*").over(janela_origem)
    ) \
    .withColumn(
        "valor_medio_origem",
        avg("amount").over(janela_origem)
    ) \
    .withColumn(
        "qtd_recebimentos_destino",
        count("*").over(janela_destino)
    ) \
    .withColumn(
        "proporcao_saldo",
        col("amount") / (col("oldbalanceOrg") + 1)
    )

df_features.createOrReplaceTempView("transacoes_features")
print("Features criadas. Veja as novas colunas:")
df_features.select(
    "type","amount","isFraud",
    "qtd_transacoes_origem",
    "valor_medio_origem",
    "qtd_recebimentos_destino",
    "proporcao_saldo"
).show(truncate=False)

"""
Freatures:

qtd_transacoes_origem — conta que nunca movimentou nada e de repente transaciona = suspeito
valor_medio_origem — transação muito acima da média histórica da conta = suspeito
qtd_recebimentos_destino — conta destino recebe de muitas contas diferentes = possível conta laranja
proporcao_saldo — transação que representa 100% do saldo disponível = padrão de fraude

Obs: Quando o "qtd_recebimentos_destino" e "proporcao_saldo", indica alta probabilidade de fraude

"""


Features criadas. Veja as novas colunas:
+--------+---------+-------+---------------------+------------------+------------------------+------------------+
|type    |amount   |isFraud|qtd_transacoes_origem|valor_medio_origem|qtd_recebimentos_destino|proporcao_saldo   |
+--------+---------+-------+---------------------+------------------+------------------------+------------------+
|CASH_OUT|269626.35|0      |1                    |269626.35         |10                      |4.323661635983589 |
|CASH_OUT|41067.19 |0      |1                    |41067.19          |10                      |0.9738715644193603|
|CASH_OUT|176515.9 |0      |1                    |176515.9          |10                      |176515.9          |
|CASH_OUT|4445.03  |0      |1                    |4445.03           |10                      |0.6320247405090288|
|CASH_OUT|111659.06|0      |1                    |111659.06         |10                      |111659.06         |
|CASH_OUT|181650.64|0      |1                  

In [ ]:
# Análise final com Spark SQL puro

spark.sql("""
    SELECT
        type,
        isFraud,
        ROUND(AVG(amount), 2) AS valor_medio,
        ROUND(AVG(proporcao_saldo), 2) AS proporcao_saldo_media,
        ROUND(AVG(qtd_recebimentos_destino), 2) AS media_recebimentos_destino,
        ROUND(AVG(qtd_transacoes_origem), 2) AS media_transacoes_origem,
        COUNT(*) AS total
    FROM transacoes_features
    GROUP BY type, isFraud
    ORDER BY type, isFraud DESC
""").show(truncate=False)

+--------+-------+-----------+---------------------+--------------------------+-----------------------+-------+
|type    |isFraud|valor_medio|proporcao_saldo_media|media_recebimentos_destino|media_transacoes_origem|total  |
+--------+-------+-----------+---------------------+--------------------------+-----------------------+-------+
|CASH_OUT|1      |1455102.59 |1197.38              |8.19                      |1.0                    |4116   |
|CASH_OUT|0      |173917.16  |77553.84             |11.21                     |1.0                    |2233384|
|TRANSFER|1      |1480891.67 |1126.38              |3.2                       |1.0                    |4097   |
|TRANSFER|0      |906229.01  |498630.59            |13.23                     |1.0                    |528812 |
+--------+-------+-----------+---------------------+--------------------------+-----------------------+-------+



In [39]:
# Preparando as features para o modelo de ML

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import when

#Convertendo type para número (ML não aceita string)
df_model = df_features.withColumn(
    "type_num",
    when(col("type") == "CASH_OUT", 1).otherwise(0)
)

# Quais features entram no modelo?
features = [
    "type_num",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "erro_saldo_orig",
    "qtd_transacoes_origem",
    "valor_medio_origem",
    "qtd_recebimentos_destino",
    "proporcao_saldo"
]

# Vector Assembler - Junta todas as ferramentas em um vetor
assembler = VectorAssembler(
    inputCols=features,
    outputCol="features_vec"
    )
df_assembled = assembler.transform(df_model)

print("Features montadas")
df_assembled.select("features_vec","isFraud").show(5,truncate=False)

"""Aqui o modelo de ML vai aprender o peso de cada uma das features: valor alto,
conta com 1 transação, conta destino recebendo de muitos, etc. Cada linha tem 11
features com valores transformados em números."""

Features montadas
+--------------------------------------------------------------------------------------------+-------+
|features_vec                                                                                |isFraud|
+--------------------------------------------------------------------------------------------+-------+
|[1.0,262774.46,0.0,0.0,359532.75,622307.21,262774.46,1.0,262774.46,5.0,262774.46]           |0      |
|[1.0,41223.87,136.0,0.0,172471.45,213695.33,41087.87,1.0,41223.87,5.0,300.90416058394163]   |0      |
|[0.0,154007.44,1448.0,0.0,18464.02,172471.45,152559.44,1.0,154007.44,5.0,106.28532781228434]|0      |
|[1.0,18464.02,40722.0,22257.98,0.0,18464.02,0.0,1.0,18464.02,5.0,0.4534052009920684]        |0      |
|[1.0,481674.22,620.0,0.0,213695.33,695369.55,481054.22,1.0,481674.22,5.0,775.6428663446054] |0      |
+--------------------------------------------------------------------------------------------+-------+
only showing top 5 rows


'Aqui o modelo de ML vai aprender o peso de cada uma das features: valor alto,\nconta com 1 transação, conta destino recebendo de muitos, etc. Cada linha tem 11\nfeatures com valores transformados em números.'

In [40]:
# Dividindo os dados e tratando o balanceamento
# Anteriormente detectamos que 0,0019% eram fraude. Iremos corrigir isso agora.
from pyspark.sql.functions import lit

# Divindo em 80% treino e 20% teste
df_treino, df_teste = df_assembled.randomSplit([0.8,0.2], seed=42)
print(f"Treino: {df_treino.count():,} transações")
print(f"Teste:  {df_teste.count():,} transações")
print(f"Fraudes no treino: {df_treino.filter(col('isFraud')==1).count():,}")

# Calculando o peso para balancear a classe
total_treino = df_treino.count()
total_fraudes = df_treino.filter(col('isFraud')==1).count()
total_legitimas = df_treino.filter(col('isFraud')==0).count()

# Peso inversamente proporcional a frequência de cada classe
peso_fraude = total_treino / (2 * total_fraudes)
peso_legitima = total_treino / (2 * total_legitimas)

print(f"\nPeso fraude: {peso_fraude:.2f}")
print(f"Peso legitima: {peso_legitima:.2f}")

# Adicionando peso como coluna
df_treino = df_treino.withColumn(
    "peso",
    when(col("isFraud") == 1, peso_fraude).otherwise(peso_legitima)
)

"""Ao calcular os pesos, ensinamos o modelo a não ignorar as fraudes, afinal, chutar "legítima" sempre dá 99% de acerto"""

Treino: 2,216,364 transações
Teste:  554,045 transações
Fraudes no treino: 6,601

Peso fraude: 167.88
Peso legitima: 0.50


'Ao calcular os pesos, ensinamos o modelo a não ignorar as fraudes, afinal, chutar "legítima" sempre dá 99% de acerto'

In [42]:
# Treinando o modelo
rf = RandomForestClassifier(
    featuresCol="features_vec",
    labelCol="isFraud",
    weightCol="peso",
    numTrees=100,        # 100 árvores de decisão
    maxDepth=10,         # profundidade máxima de cada árvore
    seed=42
)

print("Treinando o modelo... (pode levar 2-3 minutos)")
modelo = rf.fit(df_treino)
print("Modelo treinado!")

# Importância de cada feature
print("\n=== Importância das Features ===")
for feature, importancia in sorted(
    zip(features, modelo.featureImportances.toArray()),
    key=lambda x: x[1],
    reverse=True
):
    barra = "█" * int(importancia * 100)
    print(f"{feature:<30} {importancia:.4f}  {barra}")

"""
=== O que o modelo descobriu?===

1 - erro_saldo_orig (39.82%) — a feature mais poderosa. Quando o saldo após a transação não fecha matematicamente, é o sinal mais forte de fraude.
2 - proporcao_saldo (32.70%) — segunda mais importante. Fraudadores transferem quase 100% do saldo disponível. As duas features respondem por 72% do poder preditivo do modelo.
3 - qtd_transacoes_origem (0.0000%) — surpresa!
Apesar do insight de que contas fraudulentas fazem só 1 transação, o modelo não achou essa feature útil. Porque no dataset, contas legítimas também fazem poucas transações —
a feature não discrimina bem o suficiente.
"""

Treinando o modelo... (pode levar 2-3 minutos)
Modelo treinado!

=== Importância das Features ===
erro_saldo_orig                0.3982  ███████████████████████████████████████
proporcao_saldo                0.3270  ████████████████████████████████
oldbalanceOrg                  0.1041  ██████████
newbalanceOrig                 0.0614  ██████
newbalanceDest                 0.0543  █████
amount                         0.0176  █
oldbalanceDest                 0.0167  █
valor_medio_origem             0.0136  █
qtd_recebimentos_destino       0.0049  
type_num                       0.0022  
qtd_transacoes_origem          0.0000  


In [43]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col

# Fazendo predições no conjunto de teste
predicoes = modelo.transform(df_teste)

# AUC-ROC — métrica principal em antifraude
avaliador = BinaryClassificationEvaluator(
    labelCol="isFraud",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = avaliador.evaluate(predicoes)
print(f"AUC-ROC: {auc:.4f}")

# Métricas detalhadas - precision, recall
print("\n=== Métricas Detalhadas ===")
predicoes.groupBy("isFraud","prediction") \
    .count() \
    .orderBy("isFraud","prediction") \
    .show()


AUC-ROC: 1.0000

=== Métricas Detalhadas ===
+-------+----------+------+
|isFraud|prediction| count|
+-------+----------+------+
|      0|       0.0|552424|
|      0|       1.0|     9|
|      1|       0.0|     5|
|      1|       1.0|  1607|
+-------+----------+------+



In [46]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

# UDF para extrair a probabilidade de fraude do vetor
extrair_prob = udf(lambda v: float(v[1]), DoubleType())

# Selecionando as colunas relevantes para análise final
df_resultado = predicoes.select(
    "type",
    "amount",
    "nameOrig",
    "nameDest",
    "oldbalanceOrg",
    "newbalanceOrig",
    "erro_saldo_orig",
    "proporcao_saldo",
    "qtd_recebimentos_destino",
    "isFraud",
    "prediction",
    extrair_prob("probability").alias("score_fraude")
).toPandas()

# Salvando como CSV
df_resultado.to_csv("/content/resultado_fraude_sentinel.csv", index=False)

print(f"Arquivo salvo! {len(df_resultado):,} transações")
print(f"Fraudes detectadas: {int(df_resultado['prediction'].sum()):,}")
print(f"Score médio de fraude: {df_resultado['score_fraude'].mean():.4f}")

# Baixando o arquivo para seu computador
from google.colab import files
files.download("/content/resultado_fraude_sentinel.csv")

Arquivo salvo! 554,045 transações
Fraudes detectadas: 1,616
Score médio de fraude: 0.0039


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>